# 02 — Semantic Segmentation: PlantVillage Leaf Disease

Pixel-level segmentation of PlantVillage leaf images into **3 classes**:
background, healthy leaf tissue, and diseased tissue.

**Approach**:
- Architecture: **U-Net** with a lightweight **MobileNetV2** encoder (via `segmentation_models_pytorch`)
- Ground truth: 3-class masks derived from paired `color/` + `segmented/` images via HSV thresholding
- Phase 1: Freeze encoder → train decoder only (fast convergence)
- Phase 2: Unfreeze all layers → fine-tune with 10× smaller learning rate
- Combined **Dice + Cross-Entropy** loss to handle severe class imbalance at pixel level
- Deliverable: per-image **disease severity percentage** (affected / total plant area)

In [ ]:
# Install dependencies and download dataset
!pip install -q segmentation-models-pytorch albumentations
!pip install -q datasets torch torchvision opencv-python matplotlib seaborn tqdm
!pip install -q kaggle
!kaggle datasets download -d abdallahalidev/plantvillage-dataset --unzip -p /content/plantvillage

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os, glob, random
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import segmentation_models_pytorch as smp
import albumentations as A

plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print(f'Device: {DEVICE}  |  AMP: {USE_AMP}')

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
CONFIG = {
    'image_size':    256,
    'batch_size':    16,
    'phase1_epochs': 5,
    'phase2_epochs': 10,
    'base_lr':       1e-4,
    'num_classes':   3,
    'encoder':       'mobilenet_v2',
    'seed':          42,
    'save_dir':      '/content/runs/segmentation',
}

DATA_COLOR = '/content/plantvillage/plantvillage dataset/color'
DATA_SEG   = '/content/plantvillage/plantvillage dataset/segmented'

random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
os.makedirs(CONFIG['save_dir'], exist_ok=True)

print('Config:', CONFIG)

## 1. Ground-truth mask generation & paired file discovery

In [ ]:
# ── Tri-class mask from colour + segmented pair ──────────────────────────────

def _largest_component(binary_mask):
    """Keep only the largest connected component."""
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary_mask, connectivity=8
    )
    if num_labels <= 1:
        return binary_mask
    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return np.where(labels == largest, 255, 0).astype(np.uint8)


def build_triclass_mask(color_bgr, seg_bgr):
    """
    Build a (H, W) mask with class labels:
        0 = background, 1 = healthy leaf, 2 = disease.

    Parameters
    ----------
    color_bgr : np.ndarray  – colour image in BGR (from cv2.imread)
    seg_bgr   : np.ndarray  – segmented image in BGR
    """
    # ---- Leaf mask from the segmented image (non-black pixels) ----
    seg_rgb   = cv2.cvtColor(seg_bgr, cv2.COLOR_BGR2RGB)
    leaf_mask = (seg_rgb.max(axis=2) > 20).astype(np.uint8) * 255
    kernel    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    leaf_mask = cv2.morphologyEx(leaf_mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    leaf_mask = _largest_component(leaf_mask)

    # ---- Healthy-tissue mask from colour image (strict green HSV) ----
    hsv = cv2.cvtColor(color_bgr, cv2.COLOR_BGR2HSV)
    lower_healthy = np.array([30, 50, 40])
    upper_healthy = np.array([85, 255, 255])
    healthy_mask  = cv2.inRange(hsv, lower_healthy, upper_healthy)
    healthy_mask  = cv2.bitwise_and(healthy_mask, healthy_mask, mask=leaf_mask)

    # ---- Compose tri-class mask ----
    mask = np.zeros(leaf_mask.shape, dtype=np.int64)   # 0 = background
    mask[leaf_mask > 0]    = 2                          # leaf but not healthy → disease
    mask[healthy_mask > 0] = 1                          # healthy leaf

    return mask

print('build_triclass_mask() defined.')

In [ ]:
# ── Discover paired files (colour ↔ segmented) ──────────────────────────────
class_names = sorted(os.listdir(DATA_COLOR))

color_paths, seg_paths, labels = [], [], []
for cls in class_names:
    color_dir = os.path.join(DATA_COLOR, cls)
    seg_dir   = os.path.join(DATA_SEG,   cls)
    if not os.path.isdir(seg_dir):
        continue
    shared = sorted(set(os.listdir(color_dir)) & set(os.listdir(seg_dir)))
    for fname in shared:
        color_paths.append(os.path.join(color_dir, fname))
        seg_paths.append(os.path.join(seg_dir, fname))
        labels.append(cls)

print(f'Total paired samples: {len(color_paths):,}')
print(f'Classes found: {len(set(labels))}')

# ── Stratified train / val split (80/20) ─────────────────────────────────────
idx = list(range(len(color_paths)))
train_idx, val_idx = train_test_split(
    idx, test_size=0.2, stratify=labels, random_state=CONFIG['seed']
)

train_color = [color_paths[i] for i in train_idx]
train_seg   = [seg_paths[i]   for i in train_idx]
val_color   = [color_paths[i] for i in val_idx]
val_seg     = [seg_paths[i]   for i in val_idx]

print(f'Train: {len(train_color):,}  |  Val: {len(val_color):,}')

## 2. Dataset & data augmentation

In [ ]:
# ── Albumentations pipelines ─────────────────────────────────────────────────
IMG_SIZE = CONFIG['image_size']

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
])

print('Augmentation pipelines ready.')

In [ ]:
# ── Custom PyTorch Dataset ───────────────────────────────────────────────────

class PlantVillageSegDataset(Dataset):
    """Reads colour + segmented pair, builds 3-class mask on the fly."""

    def __init__(self, color_paths, seg_paths, augmentation=None):
        self.color_paths  = color_paths
        self.seg_paths    = seg_paths
        self.augmentation = augmentation

    def __len__(self):
        return len(self.color_paths)

    def __getitem__(self, i):
        # Read images (BGR)
        color_bgr = cv2.imread(self.color_paths[i])
        seg_bgr   = cv2.imread(self.seg_paths[i])

        # Build ground-truth mask (H, W) with {0, 1, 2}
        mask = build_triclass_mask(color_bgr, seg_bgr)

        # Convert colour image to RGB for augmentation / model input
        image = cv2.cvtColor(color_bgr, cv2.COLOR_BGR2RGB)

        # Apply spatial + colour augmentations to both image and mask
        if self.augmentation:
            sample = self.augmentation(image=image, mask=mask)
            image, mask = sample['image'], sample['mask']

        # To tensor: (H, W, C) uint8 → (C, H, W) float32 [0, 1]
        image = torch.from_numpy(image).float().permute(2, 0, 1) / 255.0
        mask  = torch.from_numpy(np.asarray(mask)).long()

        return image, mask

print('PlantVillageSegDataset defined.')

In [ ]:
# ── DataLoaders ──────────────────────────────────────────────────────────────
train_ds = PlantVillageSegDataset(train_color, train_seg, augmentation=train_transform)
val_ds   = PlantVillageSegDataset(val_color,   val_seg,   augmentation=val_transform)

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'], shuffle=False,
                          num_workers=2, pin_memory=True)

# Quick sanity check
imgs, masks = next(iter(train_loader))
print(f'Batch images : {imgs.shape}  (dtype={imgs.dtype})')
print(f'Batch masks  : {masks.shape}  (dtype={masks.dtype})')
print(f'Mask classes  : {sorted(masks.unique().tolist())}')

## 3. Model: U-Net + MobileNetV2

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────────
model = smp.Unet(
    encoder_name=CONFIG['encoder'],
    encoder_weights='imagenet',
    in_channels=3,
    classes=CONFIG['num_classes'],
)
model.to(DEVICE)

total_params   = sum(p.numel() for p in model.parameters())
encoder_params = sum(p.numel() for p in model.encoder.parameters())
decoder_params = total_params - encoder_params
print(f'Total params   : {total_params:,}')
print(f'Encoder params : {encoder_params:,}')
print(f'Decoder params : {decoder_params:,}')

## 4. Loss function, optimiser & training loop

In [ ]:
# ── Loss: Dice + Cross-Entropy (handles class imbalance at pixel level) ─────
dice_loss = smp.losses.DiceLoss(mode='multiclass')
ce_loss   = nn.CrossEntropyLoss()

def combined_loss(pred, target):
    return dice_loss(pred, target) + ce_loss(pred, target)

print('Combined Dice + CE loss defined.')

In [ ]:
# ── Training & validation helpers ────────────────────────────────────────────
scaler = torch.amp.GradScaler(enabled=USE_AMP)


def train_one_epoch(model, loader, optimizer):
    model.train()
    running_loss = 0.0
    for images, masks in tqdm(loader, desc='  train', leave=False):
        images = images.to(DEVICE)
        masks  = masks.to(DEVICE)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            preds = model(images)
            loss  = combined_loss(preds, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


@torch.no_grad()
def validate(model, loader):
    model.eval()
    running_loss = 0.0
    for images, masks in tqdm(loader, desc='  val  ', leave=False):
        images = images.to(DEVICE)
        masks  = masks.to(DEVICE)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            preds = model(images)
            loss  = combined_loss(preds, masks)
        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

print('Training helpers defined.')

In [ ]:
# ── Phase 1: Freeze encoder — train decoder only ────────────────────────────
for p in model.encoder.parameters():
    p.requires_grad = False

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                  lr=CONFIG['base_lr'])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['phase1_epochs'])

history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
best_model_path = os.path.join(CONFIG['save_dir'], 'best_seg_model.pth')

print(f'=== Phase 1: {CONFIG["phase1_epochs"]} epochs (encoder frozen) ===')
for epoch in range(1, CONFIG['phase1_epochs'] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_loss   = validate(model, val_loader)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    tag = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        tag = ' ★ saved'

    print(f'  Epoch {epoch}/{CONFIG["phase1_epochs"]}  '
          f'train_loss={train_loss:.4f}  val_loss={val_loss:.4f}{tag}')

In [ ]:
# ── Phase 2: Unfreeze all — fine-tune with lower LR ─────────────────────────
for p in model.encoder.parameters():
    p.requires_grad = True

optimizer = AdamW(model.parameters(), lr=CONFIG['base_lr'] / 10)
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['phase2_epochs'])

print(f'\n=== Phase 2: {CONFIG["phase2_epochs"]} epochs (all layers unfrozen, LR /10) ===')
for epoch in range(1, CONFIG['phase2_epochs'] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer)
    val_loss   = validate(model, val_loader)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    tag = ''
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        tag = ' ★ saved'

    print(f'  Epoch {epoch}/{CONFIG["phase2_epochs"]}  '
          f'train_loss={train_loss:.4f}  val_loss={val_loss:.4f}{tag}')

print(f'\nBest val loss: {best_val_loss:.4f}')

In [ ]:
# ── Loss curve ───────────────────────────────────────────────────────────────
epochs = range(1, len(history['train_loss']) + 1)
phase1_end = CONFIG['phase1_epochs']

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs, history['train_loss'], label='Train loss', color='steelblue')
ax.plot(epochs, history['val_loss'],   label='Val loss',   color='coral')
ax.axvline(phase1_end + 0.5, ls='--', color='grey', alpha=0.6, label='Unfreeze encoder')
ax.set_xlabel('Epoch')
ax.set_ylabel('Dice + CE Loss')
ax.set_title('Training & Validation Loss', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['save_dir'], 'loss_curve.png'), bbox_inches='tight')
plt.show()

## 5. Inference & disease severity quantification

In [ ]:
# ── Load best weights ────────────────────────────────────────────────────────
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE, weights_only=True))
model.eval()
print('Best model loaded.')

In [ ]:
# ── Prediction + quantification helper ───────────────────────────────────────

@torch.no_grad()
def predict_and_quantify(model, image_tensor, device):
    """
    Run inference on a single image and compute disease severity.

    Returns
    -------
    pred_mask : np.ndarray (H, W) with {0, 1, 2}
    metrics   : dict with pixel counts and severity %
    """
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device)

    with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
        output = model(image_tensor)

    pred_mask = torch.argmax(output.squeeze(0), dim=0).cpu().numpy()

    pixels_bg      = int(np.sum(pred_mask == 0))
    pixels_healthy  = int(np.sum(pred_mask == 1))
    pixels_disease  = int(np.sum(pred_mask == 2))
    total_plant     = pixels_healthy + pixels_disease

    severity = (pixels_disease / total_plant * 100) if total_plant > 0 else 0.0

    metrics = {
        'pixels_background': pixels_bg,
        'pixels_healthy':    pixels_healthy,
        'pixels_disease':    pixels_disease,
        'total_plant_area':  total_plant,
        'severity_pct':      round(severity, 2),
    }
    return pred_mask, metrics

print('predict_and_quantify() defined.')

In [ ]:
# ── Batch inference on validation samples ────────────────────────────────────
N_SAMPLES = 8
rng = random.Random(CONFIG['seed'])
sample_indices = rng.sample(range(len(val_ds)), k=min(N_SAMPLES, len(val_ds)))

sample_images, sample_gt, sample_preds, sample_metrics = [], [], [], []

for idx in sample_indices:
    img_tensor, gt_mask = val_ds[idx]
    pred_mask, metrics  = predict_and_quantify(model, img_tensor, DEVICE)

    sample_images.append(img_tensor)
    sample_gt.append(gt_mask.numpy())
    sample_preds.append(pred_mask)
    sample_metrics.append(metrics)

    print(f'Sample {idx:>5d}  |  Severity: {metrics["severity_pct"]:6.2f}%  '
          f'|  Plant area: {metrics["total_plant_area"]:,} px  '
          f'|  Diseased: {metrics["pixels_disease"]:,} px')

## 6. Visualisation: original → ground truth → prediction

In [ ]:
# ── Side-by-side comparison ──────────────────────────────────────────────────
MASK_CMAP = ListedColormap(['black', '#4CAF50', '#F44336'])   # bg, healthy, disease
legend_patches = [
    mpatches.Patch(color='black',   label='Background'),
    mpatches.Patch(color='#4CAF50', label='Healthy leaf'),
    mpatches.Patch(color='#F44336', label='Disease'),
]

n_rows = len(sample_images)
fig, axes = plt.subplots(n_rows, 3, figsize=(12, 3.5 * n_rows))
if n_rows == 1:
    axes = axes[np.newaxis, :]

for r in range(n_rows):
    # Original image (de-normalise from tensor)
    img_np = sample_images[r].permute(1, 2, 0).numpy()
    img_np = np.clip(img_np, 0, 1)

    axes[r, 0].imshow(img_np)
    axes[r, 0].set_title('Original', fontsize=10, fontweight='bold')
    axes[r, 0].axis('off')

    axes[r, 1].imshow(sample_gt[r], cmap=MASK_CMAP, vmin=0, vmax=2)
    axes[r, 1].set_title('Ground Truth', fontsize=10, fontweight='bold')
    axes[r, 1].axis('off')

    sev = sample_metrics[r]['severity_pct']
    axes[r, 2].imshow(sample_preds[r], cmap=MASK_CMAP, vmin=0, vmax=2)
    axes[r, 2].set_title(f'Prediction  (severity {sev:.1f}%)',
                         fontsize=10, fontweight='bold')
    axes[r, 2].axis('off')

fig.legend(handles=legend_patches, loc='lower center', ncol=3,
           fontsize=10, frameon=True, bbox_to_anchor=(0.5, -0.01))
fig.suptitle('Semantic Segmentation Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['save_dir'], 'segmentation_results.png'),
            bbox_inches='tight')
plt.show()

## 7. Summary: disease severity distribution

In [ ]:
# ── Severity across a larger set of validation images ────────────────────────
MAX_EVAL = min(200, len(val_ds))
eval_indices = rng.sample(range(len(val_ds)), k=MAX_EVAL)

severities = []
for idx in tqdm(eval_indices, desc='Inference'):
    img_tensor, _ = val_ds[idx]
    _, metrics = predict_and_quantify(model, img_tensor, DEVICE)
    severities.append(metrics['severity_pct'])

severities = np.array(severities)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(severities, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(np.median(severities), color='coral', ls='--',
                label=f'Median {np.median(severities):.1f}%')
axes[0].set_xlabel('Disease Severity (%)')
axes[0].set_ylabel('Count')
axes[0].set_title('Severity Distribution', fontweight='bold')
axes[0].legend()

# Box plot
sns.boxplot(x=severities, ax=axes[1], color='steelblue')
axes[1].set_xlabel('Disease Severity (%)')
axes[1].set_title('Severity Spread', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['save_dir'], 'severity_distribution.png'),
            bbox_inches='tight')
plt.show()

print(f'\nSeverity stats (n={MAX_EVAL}):')
print(f'  Mean   : {severities.mean():.2f}%')
print(f'  Median : {np.median(severities):.2f}%')
print(f'  Std    : {severities.std():.2f}%')
print(f'  Min    : {severities.min():.2f}%')
print(f'  Max    : {severities.max():.2f}%')

---

**Summary**: A U-Net with MobileNetV2 encoder was fine-tuned in two phases to
segment PlantVillage leaf images into background, healthy tissue, and diseased
tissue. The pixel-level predictions are converted into a **disease severity
percentage** — a practical metric that translates the deep-learning output into
an actionable agronomic indicator.

**Key design decisions**:
- *Lightweight encoder* — MobileNetV2 allows real-time inference on edge devices
- *Combined Dice + CE loss* — addresses the heavy class imbalance (disease pixels ≪ background)
- *HSV-derived ground truth* — avoids the need for manual pixel-level annotation
- *Two-phase training* — leverages ImageNet features while adapting to plant imagery